# 002 · Parameters and Arguments

> **Syntax › Functions** · Python 3.14

Python's parameter system looks like C#'s optional and named arguments at first glance, but it is more flexible, and it has one famous trap: mutable default values.

**In this notebook you will learn:**

- positional vs keyword arguments, and how they compare to C# named arguments
- default parameter values vs C# optional parameters
- the mutable default argument gotcha and the `None`-sentinel fix
- keyword-only (`*`) and positional-only (`/`) parameters
- the ordering rules for parameters and arguments
- why there is no `ref`/`out`, and what to do instead

## C# mental model

| C# | Python |
|---|---|
| `void F(int a, int b = 10)` | `def f(a, b=10):` |
| named argument `F(b: 2, a: 1)` | keyword argument `f(b=2, a=1)` |
| default must be a compile-time constant | default can be **any expression**, evaluated **once** |
| no way to *force* named arguments | keyword-only parameters after `*` |
| no way to *forbid* named arguments | positional-only parameters before `/` |
| `ref` / `out` parameters | not available: return a tuple |
| `params int[]` | `*args` *(see Functions › 003 · Args and Kwargs)* |

## Positional and keyword arguments

Every ordinary Python parameter can be passed **either** by position **or** by name. This is the same idea as C# named arguments, and the same rule applies: once you start passing by name, the remaining arguments must be named too.

```csharp
// C#
static string Describe(string name, int qty, decimal price) => $"{qty} x {name} @ {price}";

Describe("pen", 3, 1.5m);                    // positional
Describe(price: 1.5m, name: "pen", qty: 3);  // named, any order
Describe("pen", price: 1.5m, qty: 3);        // mixed
```

In [ ]:
def describe(name, qty, price):
    return f"{qty} x {name} @ {price}"


print(describe("pen", 3, 1.5))                  # positional
print(describe(price=1.5, name="pen", qty=3))   # keyword, any order
print(describe("pen", price=1.5, qty=3))        # mixed

Mistakes are caught at **call time** (C# catches them at compile time). Passing positional after keyword is a `SyntaxError`, so we compile that one from a string to demonstrate it safely:

In [ ]:
try:
    describe("pen", 3, qty=5)        # qty given twice
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

try:
    describe("pen", 3)               # missing price
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

try:
    compile('describe(name="pen", 3, 1.5)', "<demo>", "exec")
except SyntaxError as e:
    print(f"{type(e).__name__}: {e.msg}")

## Default values

Parameters with defaults make an argument optional, like C# optional parameters. As in C#, parameters with defaults must come after the ones without.

```csharp
// C#
static string Greet(string name, string greeting = "Hello") => $"{greeting}, {name}!";
```

In [ ]:
def greet(name, greeting="Hello", punctuation="!"):
    return f"{greeting}, {name}{punctuation}"


print(greet("alice"))
print(greet("bob", "Hi"))
print(greet("carol", punctuation="?"))    # skip the middle one by naming the last

A big difference: C# defaults must be compile-time constants (`= 10`, `= null`, `= default`). Python defaults can be **any expression**, but that expression is evaluated **once, when the `def` statement runs**, not on each call.

In [ ]:
import time

def stamp(when=time.time()):     # evaluated ONCE, at definition time
    return when


first = stamp()
time.sleep(0.01)
second = stamp()
print(first == second)           # True: the "current time" is frozen

## ⚠️ Gotcha: mutable default arguments

This is the most famous Python trap. Because the default is created once, a mutable default (list, dict, set) is **shared between all calls** that don't pass that argument.

In [ ]:
def add_item(item, items=[]):    # one list, created at def time
    items.append(item)
    return items


print(add_item("a"))   # ['a']
print(add_item("b"))   # ['a', 'b']  <- surprise: the same list again
print(add_item("c"))   # ['a', 'b', 'c']
print(add_item.__defaults__)     # the shared list lives on the function object

The fix is the **`None` sentinel** pattern: default to `None` and create a fresh object inside the function. This is exactly the C# idiom `List<string>? items = null` followed by `items ??= new()`.

```csharp
// C#
static List<string> AddItem(string item, List<string>? items = null)
{
    items ??= [];
    items.Add(item);
    return items;
}
```

In [ ]:
def add_item(item, items=None):
    if items is None:
        items = []               # a NEW list on every call
    items.append(item)
    return items


print(add_item("a"))
print(add_item("b"))
print(add_item("c", ["x"]))      # caller-supplied list still works

If `None` is itself a meaningful value for the caller, use a private sentinel object instead:

In [ ]:
_MISSING = object()   # unique object nobody else can pass by accident

def get_setting(settings, key, default=_MISSING):
    if key in settings:
        return settings[key]
    if default is _MISSING:
        raise KeyError(key)
    return default


print(get_setting({"debug": None}, "debug"))       # None is a real value here
print(get_setting({}, "debug", default=False))
try:
    get_setting({}, "debug")
except KeyError as e:
    print(f"{type(e).__name__}: {e}")

## Keyword-only parameters (`*`)

Parameters after a bare `*` **must** be passed by name. Use this for flags and options, so call sites read clearly (`retry=True` instead of a mystery `True`). C# has no equivalent: you can only *encourage* named arguments.

In [ ]:
def connect(host, port, *, timeout=5.0, retry=False):
    return f"{host}:{port} timeout={timeout} retry={retry}"


print(connect("localhost", 8080, retry=True))

try:
    connect("localhost", 8080, 10.0, True)    # positional flags are rejected
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

Keyword-only parameters do not need defaults. A keyword-only parameter without a default is simply *required and must be named*:

In [ ]:
def transfer(amount, *, source, target):
    return f"move {amount} from {source} to {target}"


print(transfer(100, source="A", target="B"))

## Positional-only parameters (`/`)

Parameters before a `/` **cannot** be passed by name. Many built-ins work this way, e.g. `len(obj=[1])` fails. Use it when the parameter name is meaningless to callers, or when you want freedom to rename it later without breaking anyone.

In [ ]:
def distance(x, y, /):
    return (x ** 2 + y ** 2) ** 0.5


print(distance(3, 4))
try:
    distance(x=3, y=4)
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

## Argument order rules

The full parameter order is:

```python
def f(pos_only, /, normal, *, kw_only): ...
#     |__________|  |______|  |_______|
#     position only  either   name only
```

With `*args`/`**kwargs` *(see Functions › 003 · Args and Kwargs)* the complete form is `def f(a, /, b, *args, c, **kwargs)`. At a call site: positional arguments first, then keyword arguments.

In [ ]:
def full(a, b=2, /, c=3, *, d, e=5):
    return f"a={a} b={b} c={c} d={d} e={e}"


print(full(1, d=4))
print(full(1, 20, 30, d=40))
print(full(1, c=30, d=40, e=50))

You can inspect any function's signature at runtime with the `inspect` module, which is a handy way to check what kind each parameter is:

In [ ]:
import inspect

for name, param in inspect.signature(full).parameters.items():
    print(f"{name}: {param.kind.name}")

## No `ref` or `out`: return a tuple instead

Python always passes **references to objects** (sometimes called "pass by object reference"). Rebinding a parameter inside the function never affects the caller's variable, so there is nothing like C# `ref`/`out`. Mutating a mutable object *is* visible to the caller, just as mutating a `List<T>` passed to a C# method would be *(see Data Types › 007 · Mutability and Identity)*.

In [ ]:
def rebind(x):
    x = 99            # rebinds the local name only


def mutate(lst):
    lst.append(99)    # changes the shared object


n = 1
rebind(n)
print(n)              # 1: unchanged

data = [1, 2]
mutate(data)
print(data)           # [1, 2, 99]: the caller sees the mutation

The idiomatic replacement for `out` parameters is returning a tuple, and for `bool TryX(..., out T value)` returning the value or `None`:

```csharp
// C#
if (int.TryParse("42", out var number))
    Console.WriteLine(number);

static void Divide(int a, int b, out int quotient, out int remainder) { ... }
```

In [ ]:
def try_parse_int(text: str) -> int | None:
    try:
        return int(text)
    except ValueError:
        return None


if (number := try_parse_int("42")) is not None:
    print(number)
print(try_parse_int("abc"))

quotient, remainder = divmod(17, 5)    # built-in that returns a tuple
print(quotient, remainder)

⚠️ **Gotcha:** watch out for falsy-but-valid results. `if (n := try_parse_int("0")):` would skip the valid value `0`. Compare against `None` explicitly *(see Data Types › 004 · Booleans and Truthiness)*.

## Summary

| Concept | C# | Python |
|---|---|---|
| Named argument | `F(qty: 3)` | `f(qty=3)` |
| Optional parameter | `int b = 10` (constant only) | `b=10` (any expression, evaluated once) |
| Mutable default | `List<T>? x = null; x ??= [];` | `x=None` then `if x is None: x = []` |
| Force named arguments | not possible | params after `*` |
| Forbid named arguments | not possible | params before `/` |
| Parameter order | required, optional, `params` | `pos_only, /, normal, *args, kw_only, **kwargs` |
| `out` / multiple results | `out int q, out int r` | `return q, r` |
| `TryParse` pattern | `bool TryX(..., out T v)` | return `T \| None` |

## Exercises

1. Fix this function so each call gets its own dict: `def tally(word, counts={}): counts[word] = counts.get(word, 0) + 1; return counts`.
2. Write `make_user(name, *, admin=False, active=True)` and show that `make_user("alice", True)` raises `TypeError`.
3. Write `clamp(value, /, low=0, high=100)` and call it with `low`/`high` by keyword. Why might `value` be positional-only?

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1 — None sentinel
def tally(word, counts=None):
    if counts is None:
        counts = {}
    counts[word] = counts.get(word, 0) + 1
    return counts

# 2 — keyword-only flags
def make_user(name, *, admin=False, active=True):
    return {"name": name, "admin": admin, "active": active}

try:
    make_user("alice", True)
except TypeError as e:
    print(e)   # takes 1 positional argument but 2 were given

# 3 — the name "value" adds nothing at the call site, and keeping it
#     positional-only lets you rename it later without breaking callers.
def clamp(value, /, low=0, high=100):
    return max(low, min(high, value))

print(clamp(150, high=120))   # 120
```

</details>